In [12]:
import os
import time
import random
from sqlalchemy import create_engine, text

import ta
import joblib
from tqdm import tqdm

# Data Collection Libraries
import yfinance as yf

# Data Preprocessing Libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Traning Model Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import requests

In [4]:
# --- Database Configuration ---
DB_USER = 'postgres'
DB_PASSWORD = 'root' # Change if your local password is different
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'converse_db' # Pointing to the new MAS database
SCHEMA_NAME = 'ml_pipeline' # The isolated bubble for training data

engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

# ==========================================
# 1 DATA INGESTION & ISOLATION
# ==========================================

In [ ]:
def setup_database():
    """Creates the isolated schema if it doesn't exist."""
    with engine.connect() as conn:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME};"))
        conn.commit()
    print(f"✅ Schema '{SCHEMA_NAME}' is ready and isolated in {DB_NAME}.")

def get_nifty50_tickers():
    """Scrapes Nifty 50 from Wikipedia."""
    print("🌍 Fetching live Nifty 50 ticker list...")
    url = 'https://en.wikipedia.org/wiki/NIFTY_50'
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    tables = pd.read_html(url, storage_options=headers)
    nifty_table = None

    for table in tables:
        if 'Symbol' in table.columns:
            nifty_table = table
            break
    
    if nifty_table is None:
        raise ValueError("Could not find the Nifty 50 constituents table with a 'Symbol' column.")
    
    tickers = nifty_table['Symbol'].astype(str) + '.NS'
    return tickers.tolist()

def fetch_intraday_data(ticker_list):
    """
    Fetches 60 days of 15-minute intraday data for all tickers 
    and pushes to the unified raw_intraday_prices table.
    """
    print(f"\nStarting data ingestion for {len(ticker_list)} tickers...")
    
    try:
        # Quick test to ensure the global engine is connected
        engine.connect()
        print(f"Successfully connected to {DB_NAME}!")
    except Exception as e:
        print(f"Database Connection Failed: {e}")
        return

    all_data = []
    
    for ticker in tqdm(ticker_list, desc="Downloading Intraday History"):
        try:
            # Random sleep between 1 to 3 seconds to trick Yahoo's scraper detection
            time.sleep(random.uniform(1.0, 3.0)) 
            
            stock = yf.Ticker(ticker)
            df = stock.history(period="60d", interval="15m")
            
            if df.empty:
                continue
                
            df.reset_index(inplace=True)
            
            # Clean timestamps
            if 'Datetime' in df.columns:
                df['Datetime'] = pd.to_datetime(df['Datetime']).dt.tz_localize(None)
                df.rename(columns={'Datetime': 'date_time'}, inplace=True)
            elif 'Date' in df.columns: # Fallback safety checks
                df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
                df.rename(columns={'Date': 'date_time'}, inplace=True)
            
            df['ticker'] = ticker
            
            # Flatten MultiIndex if yfinance returns it
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
                
            df.rename(columns={
                'Open': 'open',
                'High': 'high',
                'Low': 'low',
                'Close': 'close',
                'Volume': 'volume'
            }, inplace=True)
            
            df = df[['date_time', 'ticker', 'open', 'high', 'low', 'close', 'volume']]
            all_data.append(df)
            
        except Exception as e:
            print(f"⚠️ Failed {ticker}: {e}")
            
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        table_name = 'raw_intraday_prices'
        
        print(f"\n💾 Pushing {len(final_df):,} rows to PostgreSQL ({SCHEMA_NAME}.{table_name})...")
        final_df.to_sql(table_name, engine, schema=SCHEMA_NAME, if_exists='replace', index=False)
        print("✅ Data Ingestion Complete!")
    else:
        print("❌ No data fetched.")

# --- Execute Cell ---
# setup_database()
# nifty_tickers = get_nifty50_tickers()
# fetch_intraday_data(nifty_tickers)

✅ Schema 'ml_pipeline' is ready and isolated in converse_db.
🌍 Fetching live Nifty 50 ticker list...

Starting data ingestion for 51 tickers...
Successfully connected to converse_db!


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NAN.NS"}}}
$NAN.NS: possibly delisted; no price data found  (period=60d) (Yahoo error = "No data found, symbol may be delisted")



💾 Pushing 69,500 rows to PostgreSQL (ml_pipeline.raw_intraday_prices)...
✅ Data Ingestion Complete!


# ==========================================
# 2 ALGORITHMIC FEATURE ENGINEERING
# ==========================================

In [ ]:
def calculate_intraday_features(df):
    """
    Applies feature engineering tuned for 15-minute intraday data, 
    including new Macro Trend Anchors and Dual-Boundary Targets.
    """
    df = df.sort_values('date_time')
    
    # ==========================================
    # LAYER 1: Micro-Structure (Candle Physics)
    # ==========================================
    df['return_15m'] = df['close'].pct_change()
    df['candle_body'] = np.abs(df['close'] - df['open']) / df['open']
    df['upper_wick'] = (df['high'] - df[['open', 'close']].max(axis=1)) / df['open']
    df['lower_wick'] = (df[['open', 'close']].min(axis=1) - df['low']) / df['open']
    
    # ==========================================
    # LAYER 2: Rapid Intraday Trend
    # ==========================================
    df['ema_9'] = ta.trend.ema_indicator(df['close'], window=9)
    df['ema_21'] = ta.trend.ema_indicator(df['close'], window=21)
    
    # --- NEW: MACRO TREND ANCHORS ---
    # Moving Average Ratio (Provides continuous structural context)
    df['ema_9_21_ratio'] = df['ema_9'] / df['ema_21']
    
    # Distance from the Daily Open (Prevents blindness during slow bleeds)
    df['date_only'] = df['date_time'].dt.date
    df['daily_open'] = df.groupby('date_only')['open'].transform('first')
    df['dist_from_daily_open'] = (df['close'] - df['daily_open']) / df['daily_open']
    df.drop(columns=['date_only', 'daily_open'], inplace=True)

    # ==========================================
    # LAYER 3: Momentum & Volatility
    # ==========================================
    df['rsi_14'] = ta.momentum.rsi(df['close'], window=14)
    df['roc_4'] = ta.momentum.roc(df['close'], window=4) 
    
    macd = ta.trend.MACD(df['close'], window_fast=12, window_slow=26, window_sign=9)
    df['macd_hist'] = macd.macd_diff()
    
    df['atr_14'] = ta.volatility.average_true_range(df['high'], df['low'], df['close'], window=14)
    bollinger = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2)
    df['bollinger_width'] = bollinger.bollinger_wband()
    df['bollinger_position'] = bollinger.bollinger_pband()
    
    # ==========================================
    # LAYER 4: Volume & Time Context
    # ==========================================
    df['volume_spike'] = df['volume'] / df['volume'].rolling(window=10).mean()
    df['vwap'] = ta.volume.volume_weighted_average_price(df['high'], df['low'], df['close'], df['volume'])
    df['vwap_distance'] = (df['close'] - df['vwap']) / df['vwap']
    df.drop(columns=['vwap'], inplace=True) 

    df['hour'] = df['date_time'].dt.hour
    df['minute'] = df['date_time'].dt.minute

    # ==========================================
    # LAYER 5: Intraday Extremes & Micro-Pivots
    # ==========================================
    df['open_equals_low'] = (df['open'] == df['low']).astype(int)
    df['open_equals_high'] = (df['open'] == df['high']).astype(int)
    
    # Micro-pivots based on the LAST 15-minute candle
    df['pivot_point'] = (df['high'].shift(1) + df['low'].shift(1) + df['close'].shift(1)) / 3
    df['support_1'] = (2 * df['pivot_point']) - df['high'].shift(1)
    df['resistance_1'] = (2 * df['pivot_point']) - df['low'].shift(1)
    
    df['dist_to_pivot'] = (df['close'] - df['pivot_point']) / df['close']
    df['dist_to_s1'] = (df['close'] - df['support_1']) / df['close']
    df['dist_to_r1'] = (df['close'] - df['resistance_1']) / df['close']
    
    df.drop(columns=['pivot_point', 'support_1', 'resistance_1'], inplace=True)

    # ==========================================
    # LAYER 6: DUAL-BOUNDARY 1-HOUR TARGETS
    # ==========================================
    # We look ahead 4 candles (60 minutes) to find the absolute extremes
    df['target_high_return'] = (df['high'].rolling(window=4).max().shift(-4) - df['close']) / df['close']
    df['target_low_return'] = (df['low'].rolling(window=4).min().shift(-4) - df['close']) / df['close']

    # ==========================================
    # DATA CLEANING
    # ==========================================
    df.replace([np.inf, -np.inf], 0, inplace=True)
    
    # Drop the trailing rows that have no future 1-hour targets
    df.dropna(subset=['target_high_return', 'target_low_return'], inplace=True)
    df.fillna(0, inplace=True)
    
    return df

def run_feature_engineering():
    print("📥 Fetching raw intraday data from ml_pipeline schema...")
    raw_df = pd.read_sql(f"SELECT * FROM {SCHEMA_NAME}.raw_intraday_prices", engine)
    
    print(f"⚙️ Engineering features & dual-targets for {len(raw_df['ticker'].unique())} tickers...")
    
    # Apply calculations per stock
    processed_df = raw_df.groupby('ticker', group_keys=False).apply(calculate_intraday_features)
    
    table_name = 'processed_intraday_features'
    print(f"💾 Pushing {len(processed_df):,} prepared rows to {SCHEMA_NAME}.{table_name}...")
    
    # Write back to the isolated schema
    processed_df.to_sql(table_name, engine, schema=SCHEMA_NAME, if_exists='replace', index=False)
    print("✅ Feature Engineering Complete!")

# --- Execute Cell ---
# run_feature_engineering()

📥 Fetching raw intraday data from ml_pipeline schema...
⚙️ Engineering features & dual-targets for 50 tickers...


C:\Users\Tuf_F15\AppData\Local\Temp\ipykernel_24744\3221318592.py:99: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  processed_df = raw_df.groupby('ticker', group_keys=False).apply(calculate_intraday_features)


💾 Pushing 69,300 prepared rows to ml_pipeline.processed_intraday_features...
✅ Feature Engineering Complete!


# ==========================================
# 3 DATA VALIDATION & HEALTH CHECK
# ==========================================

In [5]:
def validate_features():
    print(f"📥 Fetching processed data from {SCHEMA_NAME}.processed_intraday_features...")
    df = pd.read_sql(f"SELECT * FROM {SCHEMA_NAME}.processed_intraday_features", engine)

    print("\n--- 1. Checking for Fatal Errors (NaNs & Infinity) ---")
    # This should be exactly 0. If it's not, PyTorch will crash.
    inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
    nan_count = df.isna().sum().sum()
    
    print(f"Total Infinity values: {inf_count}")
    if inf_count > 0:
        print("⚠️ WARNING: Infinities detected! Check division-by-zero in feature engineering.")
        
    print(f"Total NaN values: {nan_count}")
    if nan_count > 0:
        print("⚠️ WARNING: NaNs detected! Check rolling window drops.")

    print("\n--- 2. Validating Mathematical Bounds ---")
    # RSI must strictly be between 0 and 100.
    rsi_min = df['rsi_14'].min()
    rsi_max = df['rsi_14'].max()
    print(f"RSI Min: {rsi_min:.2f} (Should be >= 0)")
    print(f"RSI Max: {rsi_max:.2f} (Should be <= 100)")
    
    if rsi_min < 0 or rsi_max > 100:
        print("⚠️ WARNING: RSI bounds violated.")

    print("\n--- 3. Validating Feature Distributions ---")
    # Look at standard deviations to ensure new macro features aren't wildly skewed
    check_cols = ['dist_from_daily_open', 'ema_9_21_ratio', 'volume_spike', 'target_high_return', 'target_low_return']
    print(df[check_cols].describe().round(4))
    
    print("\n✅ Validation Complete! If NaNs and Infs are 0, you are ready for Tensor scaling.")

# --- Execute Cell ---
validate_features()

📥 Fetching processed data from ml_pipeline.processed_intraday_features...

--- 1. Checking for Fatal Errors (NaNs & Infinity) ---
Total Infinity values: 0
Total NaN values: 0

--- 2. Validating Mathematical Bounds ---
RSI Min: 0.00 (Should be >= 0)
RSI Max: 97.40 (Should be <= 100)

--- 3. Validating Feature Distributions ---
       dist_from_daily_open  ema_9_21_ratio  volume_spike  target_high_return  \
count            69300.0000      69300.0000    69300.0000          69300.0000   
mean                 0.0001          0.9857        1.0372              0.0042   
std                  0.0119          0.1193        0.7864              0.0066   
min                 -0.0880          0.0000        0.0000             -0.0839   
25%                 -0.0069          0.9980        0.5378              0.0011   
50%                 -0.0003          1.0000        0.8299              0.0028   
75%                  0.0064          1.0020        1.2824              0.0055   
max                  0.0

# ==========================================
# 4 DUAL-BOUNDARY TENSOR GENERATION
# ==========================================

In [7]:
# We will save the tensors inside the current working directory of the notebook
SAVE_DIR = os.path.join(os.getcwd(), 'data_tensors')
os.makedirs(SAVE_DIR, exist_ok=True)
WINDOW_SIZE = 60 # 60 periods = ~2.5 market days on a 15m chart

def create_dual_sequences(df, feature_cols):
    X, y = [], []
    data_features = df[feature_cols].values
    
    # --- NEW: Stack the High and Low targets together ---
    data_targets = df[['target_high_return', 'target_low_return']].values
    
    for i in range(len(df) - WINDOW_SIZE):
        X.append(data_features[i : i + WINDOW_SIZE])
        y.append(data_targets[i + WINDOW_SIZE])
        
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def build_intraday_tensors():
    print(f"📥 Loading processed features from {SCHEMA_NAME}.processed_intraday_features...")
    df = pd.read_sql(f"SELECT * FROM {SCHEMA_NAME}.processed_intraday_features", engine)
    df.sort_values(by=['ticker', 'date_time'], inplace=True)
    
    # Exclude non-features and targets
    exclude_cols = ['date_time', 'ticker', 'target_high_return', 'target_low_return']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    print(f"Total Model Features: {len(feature_cols)}")
    
    print("⏳ Performing Chronological Time-Based Split...")
    unique_dates = df['date_time'].dt.date.unique()
    unique_dates.sort()
    
    # Strict cutoff to prevent future data leakage
    test_cutoff = pd.to_datetime(unique_dates[-10])
    val_cutoff = pd.to_datetime(unique_dates[-20])
    
    train_df = df[df['date_time'] < val_cutoff].copy()
    val_df = df[(df['date_time'] >= val_cutoff) & (df['date_time'] < test_cutoff)].copy()
    test_df = df[df['date_time'] >= test_cutoff].copy()
    
    print("⚖️ Scaling Features (Fitting on Train Only)...")
    scaler = StandardScaler()
    
    train_df.loc[:, feature_cols] = scaler.fit_transform(train_df[feature_cols])
    val_df.loc[:, feature_cols] = scaler.transform(val_df[feature_cols])
    test_df.loc[:, feature_cols] = scaler.transform(test_df[feature_cols])
    
    joblib.dump(scaler, os.path.join(SAVE_DIR, 'intraday_scaler.pkl'))
    
    print(f"🔄 Generating 3D Sliding Sequences (Window = {WINDOW_SIZE})...")
    
    def process_split(split_df, split_name):
        X_list, y_list = [], []
        
        for ticker, stock_df in split_df.groupby('ticker'):
            if len(stock_df) <= WINDOW_SIZE:
                continue
                
            X_stock, y_stock = create_dual_sequences(stock_df, feature_cols)
            X_list.append(X_stock)
            y_list.append(y_stock)
            
        X_final = np.concatenate(X_list, axis=0)
        y_final = np.concatenate(y_list, axis=0)
        
        print(f"   {split_name.capitalize()} Set -> X: {X_final.shape}, y: {y_final.shape}")
        
        np.save(os.path.join(SAVE_DIR, f'X_{split_name}.npy'), X_final)
        np.save(os.path.join(SAVE_DIR, f'y_{split_name}.npy'), y_final)
        
    process_split(train_df, 'train')
    process_split(val_df, 'val')
    process_split(test_df, 'test')
    
    print(f"\n✅ Success! Dual-Boundary Tensors saved to '{SAVE_DIR}'. Ready for PyTorch.")

# --- Execute Cell ---
build_intraday_tensors()

📥 Loading processed features from ml_pipeline.processed_intraday_features...
Total Model Features: 28
⏳ Performing Chronological Time-Based Split...
⚖️ Scaling Features (Fitting on Train Only)...
🔄 Generating 3D Sliding Sequences (Window = 60)...


C:\Users\Tuf_F15\AppData\Local\Temp\ipykernel_22516\2421367756.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.2503465  -0.15725622 -0.11287728 ...  1.34140782  1.3242643
  2.83300889]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, feature_cols] = scaler.fit_transform(train_df[feature_cols])
C:\Users\Tuf_F15\AppData\Local\Temp\ipykernel_22516\2421367756.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.55567863 -1.55567863 -1.55567863 ...  1.14958977  1.69064345
  1.69064345]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_df.loc[:, feature_cols] = scaler.fit_transform(train_df[feature_cols])
C:\Users\Tuf_F15\AppData\Local\Temp\ipykernel_22516\2421367756.py:45: FutureWarning: Setting an item of incompatible dtype is depreca

   Train Set -> X: (42300, 60, 28), y: (42300, 2)
   Val Set -> X: (9500, 60, 28), y: (9500, 2)
   Test Set -> X: (8500, 60, 28), y: (8500, 2)

✅ Success! Dual-Boundary Tensors saved to 'c:\Users\Tuf_F15\Desktop\codes\Python\Project\stock_predictor\test\Intraday_Engine\Converse_Prediction_model\data_tensors'. Ready for PyTorch.


# ==========================================
# 5 DUAL-BOUNDARY ARCHITECTURE & TRAINING
# ==========================================

In [19]:
VERSION = 3

# --- Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Deploying Neural Network on: {device}")

TENSOR_DIR = os.path.join(os.getcwd(), 'data_tensors')
WEIGHTS_DIR = os.path.join(os.getcwd(), 'weights')
os.makedirs(WEIGHTS_DIR, exist_ok=True)
MODEL_SAVE_PATH = os.path.join(WEIGHTS_DIR, f'dual_intraday_attention_V{VERSION}.pth')

class StockSequenceDataset(Dataset):
    def __init__(self, x_path, y_path):
        self.X = torch.tensor(np.load(x_path), dtype=torch.float32)
        self.y = torch.tensor(np.load(y_path), dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ==========================================
# THE ATTENTION MECHANISM
# ==========================================
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_outputs):
        attn_weights = torch.softmax(self.attention(lstm_outputs), dim=1) 
        context_vector = torch.sum(attn_weights * lstm_outputs, dim=1) 
        return context_vector, attn_weights

# ==========================================
# THE DUAL-BOUNDARY ARCHITECTURE
# ==========================================
class StockCNNLSTMAttention(nn.Module):
    def __init__(self, input_dim, cnn_out_channels, kernel_size, hidden_dim, num_layers, dropout_rate):
        super(StockCNNLSTMAttention, self).__init__()
        pad = kernel_size // 2 
        
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=cnn_out_channels, kernel_size=kernel_size, padding=pad)
        self.relu = nn.ReLU()
        
        self.lstm = nn.LSTM(cnn_out_channels, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.attention = Attention(hidden_dim)
        
        # THE UPGRADE: Outputting 2 values instead of 1
        self.fc = nn.Linear(hidden_dim, 2)
        
    def forward(self, x):
        x = x.permute(0, 2, 1) 
        x = self.conv1(x)
        x = self.relu(x)
        x = x.permute(0, 2, 1) 
        
        h0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(device)
        c0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(device)
        
        lstm_out, _ = self.lstm(x, (h0, c0))
        context_vector, _ = self.attention(lstm_out)
        
        out = self.fc(context_vector)
        return out # No longer squeezing -1 because we have 2 dimensions

def train_model():
    print("📂 Loading 3D Tensors into RAM...")
    train_dataset = StockSequenceDataset(os.path.join(TENSOR_DIR, 'X_train.npy'), os.path.join(TENSOR_DIR, 'y_train.npy'))
    val_dataset = StockSequenceDataset(os.path.join(TENSOR_DIR, 'X_val.npy'), os.path.join(TENSOR_DIR, 'y_val.npy'))
    test_dataset = StockSequenceDataset(os.path.join(TENSOR_DIR, 'X_test.npy'), os.path.join(TENSOR_DIR, 'y_test.npy'))
    
    batch_size = 512
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Dynamically read how many features we engineered
    dynamic_input_dim = train_dataset.X.shape[2]
    print(f"📐 Detected Input Dimension: {dynamic_input_dim} features.")

    # # Initialize the Dual-Output Model
    # model = StockCNNLSTMAttention(
    #     input_dim=dynamic_input_dim,
    #     cnn_out_channels=128,
    #     kernel_size=5,
    #     hidden_dim=128,
    #     # num_layers=1, 
    #     # dropout_rate=0.0
    #     num_layers=2,
    #     dropout_rate=0.3
    # ).to(device)
    
    # criterion = nn.MSELoss()
    # # Slightly lowered learning rate for stable dual-regression
    # optimizer = optim.Adam(model.parameters(), lr=0.0001) 

    # --- INJECTING OPTUNA WINNERS HERE ---
    model = StockCNNLSTMAttention(
        input_dim=dynamic_input_dim, 
        cnn_out_channels=128,       # Optuna Winner
        kernel_size=5,
        hidden_dim=128,             # Optuna Winner
        num_layers=2,               # Optuna Winner
        dropout_rate=0.3881         # Optuna Winner
    ).to(device)
    
    criterion = nn.MSELoss()
    # INJECTING OPTUNA OPTIMIZED LEARNING RATE
    optimizer = optim.Adam(model.parameters(), lr=0.0004479)
    
    epochs = 25
    best_val_loss = float('inf')
    
    print("\n🔥 Commencing Dual-Target Training...")
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # Validation Evaluation
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item() * batch_X.size(0)
                
        val_loss /= len(val_loader.dataset)
        
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.7f} | Val Loss: {val_loss:.7f}")
        
        # Save the absolute best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            
    print(f"\n✅ Training Complete in {(time.time() - start_time)/60:.1f} minutes.")
    print(f"💾 Best Dual-Boundary weights frozen and saved to: {MODEL_SAVE_PATH}")

    # --- Unseen Data Test ---
    print("\n--- 🧪 FINAL UNSEEN TEST SET EVALUATION ---")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, weights_only=True))
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            test_loss += loss.item() * batch_X.size(0)
    
    test_loss /= len(test_loader.dataset)
    print(f"Unseen Test Set MSE Loss: {test_loss:.7f}")

# --- Execute Cell ---
train_model()

🚀 Deploying Neural Network on: cuda
📂 Loading 3D Tensors into RAM...
📐 Detected Input Dimension: 28 features.

🔥 Commencing Dual-Target Training...
Epoch [1/25] | Train Loss: 0.0002419 | Val Loss: 0.0000271
Epoch [2/25] | Train Loss: 0.0000457 | Val Loss: 0.0000257
Epoch [3/25] | Train Loss: 0.0000446 | Val Loss: 0.0000257
Epoch [4/25] | Train Loss: 0.0000439 | Val Loss: 0.0000251
Epoch [5/25] | Train Loss: 0.0000436 | Val Loss: 0.0000249
Epoch [6/25] | Train Loss: 0.0000433 | Val Loss: 0.0000250
Epoch [7/25] | Train Loss: 0.0000429 | Val Loss: 0.0000250
Epoch [8/25] | Train Loss: 0.0000429 | Val Loss: 0.0000256
Epoch [9/25] | Train Loss: 0.0000426 | Val Loss: 0.0000247
Epoch [10/25] | Train Loss: 0.0000426 | Val Loss: 0.0000262
Epoch [11/25] | Train Loss: 0.0000424 | Val Loss: 0.0000253
Epoch [12/25] | Train Loss: 0.0000425 | Val Loss: 0.0000253
Epoch [13/25] | Train Loss: 0.0000422 | Val Loss: 0.0000253
Epoch [14/25] | Train Loss: 0.0000419 | Val Loss: 0.0000251
Epoch [15/25] | Train

# ==========================================
# 6 LIVE INFERENCE ENGINE (MC DROPOUT)
# ==========================================

In [14]:
# --- Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SCALER_PATH = os.path.join(os.getcwd(), 'data_tensors', 'intraday_scaler.pkl')
WEIGHTS_PATH = os.path.join(os.getcwd(), 'weights', f'dual_intraday_attention_V{VERSION}.pth')

# Chrome Disguise to bypass Yahoo blocks
yf_session = requests.Session()
yf_session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'
})

def fetch_live_15m_tensor(ticker: str):
    """Downloads the last few days of 15m data and applies exact feature engineering."""
    stock = yf.Ticker(ticker, session=yf_session)
    df = stock.history(period="5d", interval="15m")
    
    if df.empty or len(df) < 65:
        raise ValueError(f"Not enough 15m data for {ticker}.")
        
    df.reset_index(inplace=True)
    date_col = 'Datetime' if 'Datetime' in df.columns else 'Date'
    df.rename(columns={date_col: 'date_time', 'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close', 'Volume': 'volume'}, inplace=True)
    df['date_time'] = pd.to_datetime(df['date_time']).dt.tz_localize(None)
    
    # 1. Physics & Trend
    df['return_15m'] = df['close'].pct_change()
    df['candle_body'] = np.abs(df['close'] - df['open']) / df['open']
    df['upper_wick'] = (df['high'] - df[['open', 'close']].max(axis=1)) / df['open']
    df['lower_wick'] = (df[['open', 'close']].min(axis=1) - df['low']) / df['open']
    df['ema_9'] = ta.trend.ema_indicator(df['close'], window=9)
    df['ema_21'] = ta.trend.ema_indicator(df['close'], window=21)
    
    # 2. Macro Anchors
    df['ema_9_21_ratio'] = df['ema_9'] / df['ema_21']
    df['date_only'] = df['date_time'].dt.date
    df['daily_open'] = df.groupby('date_only')['open'].transform('first')
    df['dist_from_daily_open'] = (df['close'] - df['daily_open']) / df['daily_open']
    df.drop(columns=['date_only', 'daily_open'], inplace=True)
    
    # 3. Momentum & Volatility
    df['rsi_14'] = ta.momentum.rsi(df['close'], window=14)
    df['roc_4'] = ta.momentum.roc(df['close'], window=4) 
    macd = ta.trend.MACD(df['close'], window_fast=12, window_slow=26, window_sign=9)
    df['macd_hist'] = macd.macd_diff()
    df['atr_14'] = ta.volatility.average_true_range(df['high'], df['low'], df['close'], window=14)
    bollinger = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2)
    df['bollinger_width'] = bollinger.bollinger_wband()
    df['bollinger_position'] = bollinger.bollinger_pband()
    
    # 4. Volume & Time
    df['volume_spike'] = df['volume'] / df['volume'].rolling(window=10).mean()
    df['vwap'] = ta.volume.volume_weighted_average_price(df['high'], df['low'], df['close'], df['volume'])
    df['vwap_distance'] = (df['close'] - df['vwap']) / df['vwap']
    df.drop(columns=['vwap'], inplace=True) 
    df['hour'] = df['date_time'].dt.hour
    df['minute'] = df['date_time'].dt.minute
    
    # 5. Micro Pivots
    df['open_equals_low'] = (df['open'] == df['low']).astype(int)
    df['open_equals_high'] = (df['open'] == df['high']).astype(int)
    df['pivot_point'] = (df['high'].shift(1) + df['low'].shift(1) + df['close'].shift(1)) / 3
    df['support_1'] = (2 * df['pivot_point']) - df['high'].shift(1)
    df['resistance_1'] = (2 * df['pivot_point']) - df['low'].shift(1)
    df['dist_to_pivot'] = (df['close'] - df['pivot_point']) / df['close']
    df['dist_to_s1'] = (df['close'] - df['support_1']) / df['close']
    df['dist_to_r1'] = (df['close'] - df['resistance_1']) / df['close']
    df.drop(columns=['pivot_point', 'support_1', 'resistance_1'], inplace=True)

    df.replace([np.inf, -np.inf], 0, inplace=True)
    df.fillna(0, inplace=True)
    
    # Extract just the features
    exclude_cols = ['date_time', 'ticker', 'Stock Splits', 'Dividends']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    scaler = joblib.load(SCALER_PATH)
    scaled_features = scaler.transform(df[feature_cols])
    
    # Grab the last 60 periods (WINDOW_SIZE)
    final_sequence = scaled_features[-60:]
    tensor_input = torch.tensor(final_sequence, dtype=torch.float32).unsqueeze(0).to(device)
    
    current_price = df['close'].iloc[-1]
    current_atr = df['atr_14'].iloc[-1]
    
    return tensor_input, current_price, current_atr

def get_dual_prediction(ticker: str):
    print(f"🧠 Running MC Dropout Inference for {ticker}...")
    
    tensor_input, current_price, current_atr = fetch_live_15m_tensor(ticker)
    
    # Initialize Architecture (Must match Cell 5)
    model = StockCNNLSTMAttention(
        input_dim=28, 
        cnn_out_channels=128, 
        kernel_size=5,
        hidden_dim=128, 
        num_layers=2,       # Now expecting 2 layers
        dropout_rate=0.3    # Now expecting Dropout
    ).to(device)
    
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device, weights_only=True))
    
    # --- ENABLE MC DROPOUT ---
    model.train() 
    
    passes = 50
    high_preds = []
    low_preds = []
    
    with torch.no_grad():
        for _ in range(passes):
            # Model outputs [batch_size, 2]. We slice the two outputs.
            out = model(tensor_input)
            high_preds.append(out[0][0].item())
            low_preds.append(out[0][1].item())
            
    # Calculate Expectations and Uncertainty
    expected_high = np.mean(high_preds)
    expected_low = np.mean(low_preds)
    
    # Uncertainty is the combined standard deviation
    uncertainty = np.std(high_preds) + np.std(low_preds)
    
    print("\n==========================================")
    print(f"🎯 1-HOUR FORECAST: {ticker}")
    print("==========================================")
    print(f"Current Price:      ₹{current_price:.2f}")
    print(f"Expected High (TP): {expected_high*100:+.3f}%  (₹{current_price * (1 + expected_high):.2f})")
    print(f"Expected Low (SL):  {expected_low*100:+.3f}%  (₹{current_price * (1 + expected_low):.2f})")
    print(f"System Uncertainty: {uncertainty:.5f}")
    print("==========================================\n")

# --- Execute Test ---
get_dual_prediction("ADANIENT.NS")

🧠 Running MC Dropout Inference for ADANIENT.NS...

🎯 1-HOUR FORECAST: ADANIENT.NS
Current Price:      ₹3065.00
Expected High (TP): +0.920%  (₹3093.21)
Expected Low (SL):  -0.477%  (₹3050.37)
System Uncertainty: 0.00145



# ==========================================
# 7 DUAL-BOUNDARY HISTORICAL BACKTESTER
# ==========================================

In [21]:
import torch
import numpy as np
import os
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TENSOR_DIR = os.path.join(os.getcwd(), 'data_tensors')
WEIGHTS_PATH = os.path.join(os.getcwd(), 'weights', f'dual_intraday_attention_V{VERSION}.pth')

def run_boundary_validation():
    print("🧪 Loading Unseen Test Set for Boundary Validation...")
    X_test = torch.tensor(np.load(os.path.join(TENSOR_DIR, 'X_test.npy')), dtype=torch.float32).to(device)
    y_test = np.load(os.path.join(TENSOR_DIR, 'y_test.npy')) 

    input_dim = X_test.shape[2]
    # Reinitialize architecture
    model = StockCNNLSTMAttention(
        input_dim=input_dim, cnn_out_channels=128, kernel_size=5,
        hidden_dim=128, num_layers=2, dropout_rate=0.3
    ).to(device)

    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device, weights_only=True))
    model.eval()

    with torch.no_grad():
        predictions = model(X_test).cpu().numpy()

    print("⚙️ Calculating Absolute Boundary Accuracy...")

    high_errors = []
    low_errors = []
    inside_bounds_count = 0

    for i in range(len(predictions)):
        pred_high = predictions[i][0]
        pred_low = predictions[i][1]
        actual_high = y_test[i][0]
        actual_low = y_test[i][1]

        # Calculate absolute error in percentage terms
        high_errors.append(abs(pred_high - actual_high) * 100)
        low_errors.append(abs(pred_low - actual_low) * 100)

        # Did the actual price stay within or touch our predicted geometry?
        # Adding a 0.2% buffer for standard market noise
        if actual_low >= (pred_low - 0.002) and actual_high <= (pred_high + 0.002):
            inside_bounds_count += 1

    mean_high_error = np.mean(high_errors)
    mean_low_error = np.mean(low_errors)
    containment_rate = (inside_bounds_count / len(predictions)) * 100

    print("\n======================================")
    print("🎯 BOUNDARY VALIDATION RESULTS 🎯")
    print("======================================")
    print(f"Total 1-Hour Windows Tested: {len(predictions):,}")
    print(f"Ceiling (TP) Average Error:  ±{mean_high_error:.3f}%")
    print(f"Floor (SL) Average Error:    ±{mean_low_error:.3f}%")
    print(f"Price Containment Rate:      {containment_rate:.2f}%")
    print("======================================")

# --- Execute Cell ---
run_boundary_validation()

🧪 Loading Unseen Test Set for Boundary Validation...
⚙️ Calculating Absolute Boundary Accuracy...

🎯 BOUNDARY VALIDATION RESULTS 🎯
Total 1-Hour Windows Tested: 8,500
Ceiling (TP) Average Error:  ±0.320%
Floor (SL) Average Error:    ±0.274%
Price Containment Rate:      63.33%


In [18]:
# ==========================================
# CELL 8: HYPERPARAMETER TUNING (OPTUNA)
# ==========================================
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import optuna
import warnings

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TENSOR_DIR = os.path.join(os.getcwd(), 'data_tensors')

class StockSequenceDataset(Dataset):
    def __init__(self, x_path, y_path):
        self.X = torch.tensor(np.load(x_path), dtype=torch.float32)
        self.y = torch.tensor(np.load(y_path), dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_outputs):
        attn_weights = torch.softmax(self.attention(lstm_outputs), dim=1) 
        context_vector = torch.sum(attn_weights * lstm_outputs, dim=1) 
        return context_vector, attn_weights

class StockCNNLSTMAttention(nn.Module):
    def __init__(self, input_dim, cnn_out_channels, kernel_size, hidden_dim, num_layers, dropout_rate):
        super(StockCNNLSTMAttention, self).__init__()
        pad = kernel_size // 2 
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=cnn_out_channels, kernel_size=kernel_size, padding=pad)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(cnn_out_channels, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.attention = Attention(hidden_dim)
        
        # UPGRADE: Map output to 2 boundaries
        self.fc = nn.Linear(hidden_dim, 2)
        
    def forward(self, x):
        x = x.permute(0, 2, 1) 
        x = self.conv1(x)
        x = self.relu(x)
        x = x.permute(0, 2, 1) 
        
        h0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(device)
        c0 = torch.zeros(self.lstm.num_layers, x.size(0), self.lstm.hidden_size).to(device)
        
        lstm_out, _ = self.lstm(x, (h0, c0))
        context_vector, _ = self.attention(lstm_out)
        return self.fc(context_vector)

def objective(trial):
    # Optuna Search Space Parameters
    cnn_channels = trial.suggest_categorical('cnn_channels', [64, 128])
    hidden_dim = trial.suggest_categorical('hidden_dim', [64, 128])
    num_layers = trial.suggest_int('num_layers', 1, 2)
    dropout_rate = trial.suggest_float('dropout', 0.1, 0.4) if num_layers > 1 else 0.0
    learning_rate = trial.suggest_float('lr', 5e-5, 1e-3, log=True)
    batch_size = 512

    train_dataset = StockSequenceDataset(os.path.join(TENSOR_DIR, 'X_train.npy'), os.path.join(TENSOR_DIR, 'y_train.npy'))
    val_dataset = StockSequenceDataset(os.path.join(TENSOR_DIR, 'X_val.npy'), os.path.join(TENSOR_DIR, 'y_val.npy'))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    dynamic_input_dim = train_dataset.X.shape[2]

    model = StockCNNLSTMAttention(
        input_dim=dynamic_input_dim, 
        cnn_out_channels=cnn_channels, 
        kernel_size=5,
        hidden_dim=hidden_dim, 
        num_layers=num_layers, 
        dropout_rate=dropout_rate
    ).to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    epochs = 8 # Kept efficient for rapid tuning runs
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item() * batch_X.size(0)
                
        val_loss /= len(val_loader.dataset)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            
        # Report back to Optuna engine for pruning steps
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_loss

if __name__ == "__main__":
    print(f"Deploying Dual-Boundary Tuning Matrix to: {device}")
    
    # We change direction to MINIMIZE because we are tuning for lowest MSE loss!
    study = optuna.create_study(direction='minimize', study_name="Dual_Boundary_Tuning")
    study.optimize(objective, n_trials=15) 
    
    print("\n======================================")
    print("🏆 OPTIMIZED HYPERPARAMETERS FOUND 🏆")
    print("======================================")
    print(f"Best Validation MSE Loss: {study.best_value:.7f}")
    for key, value in study.best_trial.params.items():
        print(f" -> {key}: {value}")

c:\Users\Tuf_F15\Desktop\codes\Python\Project\stock_predictor\stock_predictor_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-07-09 11:46:43,839] A new study created in memory with name: Dual_Boundary_Tuning


Deploying Dual-Boundary Tuning Matrix to: cuda


[I 2026-07-09 11:46:51,753] Trial 0 finished with value: 2.8031691990786626e-05 and parameters: {'cnn_channels': 64, 'hidden_dim': 64, 'num_layers': 1, 'lr': 0.0005577171356595804}. Best is trial 0 with value: 2.8031691990786626e-05.
[I 2026-07-09 11:46:59,386] Trial 1 finished with value: 2.5770098061337576e-05 and parameters: {'cnn_channels': 64, 'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.16919739455657207, 'lr': 0.0003035720296462944}. Best is trial 1 with value: 2.5770098061337576e-05.
[I 2026-07-09 11:47:06,812] Trial 2 finished with value: 2.6486573595320804e-05 and parameters: {'cnn_channels': 64, 'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.21832859846176575, 'lr': 0.00011166647313926138}. Best is trial 1 with value: 2.5770098061337576e-05.
[I 2026-07-09 11:47:23,646] Trial 3 finished with value: 2.498813738930039e-05 and parameters: {'cnn_channels': 128, 'hidden_dim': 128, 'num_layers': 2, 'dropout': 0.3801166702530472, 'lr': 0.0007220294739515476}. Best is trial 3 wit


🏆 OPTIMIZED HYPERPARAMETERS FOUND 🏆
Best Validation MSE Loss: 0.0000247
 -> cnn_channels: 128
 -> hidden_dim: 128
 -> num_layers: 2
 -> dropout: 0.38810832068327095
 -> lr: 0.00044792929843612023


In [22]:
# ==========================================
# CELL 9: SMART EXECUTION AGENT (SIMULATION)
# ==========================================
import torch
import numpy as np
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TENSOR_DIR = os.path.join(os.getcwd(), 'data_tensors')
WEIGHTS_PATH = os.path.join(os.getcwd(), 'weights', 'dual_intraday_attention_v3.pth')

def run_smart_execution():
    print("🤖 Booting Smart Execution Agent...")
    X_test = torch.tensor(np.load(os.path.join(TENSOR_DIR, 'X_test.npy')), dtype=torch.float32).to(device)
    y_test = np.load(os.path.join(TENSOR_DIR, 'y_test.npy')) 

    input_dim = X_test.shape[2]
    model = StockCNNLSTMAttention(
        input_dim=input_dim, cnn_out_channels=128, kernel_size=5,
        hidden_dim=128, num_layers=2, dropout_rate=0.3881
    ).to(device)

    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device, weights_only=True))
    model.eval()

    with torch.no_grad():
        predictions = model(X_test).cpu().numpy()
        
    print("⚙️ Scanning 8,500 market environments for asymmetric Risk/Reward...")
    
    STARTING_CAPITAL = 10000.0
    FEE_PER_TRADE = 0.0005 # 0.05% brokerage/slippage
    CAPITAL_PER_TRADE = 0.10 # Risk 10% per trade
    
    current_capital = STARTING_CAPITAL
    long_wins, long_losses = 0, 0
    short_wins, short_losses = 0, 0
    
    for i in range(len(predictions)):
        expected_high = predictions[i][0]
        expected_low = predictions[i][1]
        
        actual_high = y_test[i][0]
        actual_low = y_test[i][1]
        
        allocated_funds = current_capital * CAPITAL_PER_TRADE
        
        # --- STRATEGY 1: LONG EXECUTION ---
        # Rule: TP must be > 0.5%, and Reward must be 2x greater than Risk
        if expected_high >= 0.005 and expected_high > (abs(expected_low) * 2.0):
            # We place our Stop Loss exactly at the predicted floor
            if actual_low <= expected_low: 
                current_capital += allocated_funds * (expected_low - FEE_PER_TRADE)
                long_losses += 1
            elif actual_high >= expected_high:
                current_capital += allocated_funds * (expected_high - FEE_PER_TRADE)
                long_wins += 1

        # --- STRATEGY 2: SHORT EXECUTION ---
        # Rule: Floor drop must be > 0.5%, and Downside Reward must be 2x greater than Upside Risk
        elif abs(expected_low) >= 0.005 and abs(expected_low) > (expected_high * 2.0):
            # Shorting means we win if it hits the floor, and lose if it hits the ceiling
            if actual_high >= expected_high: 
                # Stop Loss hit (price went up)
                current_capital += allocated_funds * (-expected_high - FEE_PER_TRADE)
                short_losses += 1
            elif actual_low <= expected_low:
                # Take Profit hit (price went down)
                current_capital += allocated_funds * (abs(expected_low) - FEE_PER_TRADE)
                short_wins += 1

    # --- Results ---
    total_trades = long_wins + long_losses + short_wins + short_losses
    win_rate = ((long_wins + short_wins) / total_trades * 100) if total_trades > 0 else 0
    roi = ((current_capital - STARTING_CAPITAL) / STARTING_CAPITAL) * 100
    
    print("\n======================================")
    print("📈 ASYMMETRIC EXECUTION RESULTS 📈")
    print("======================================")
    print(f"Total Trades Taken: {total_trades}")
    print(f"Longs (W/L):        {long_wins} / {long_losses}")
    print(f"Shorts (W/L):       {short_wins} / {short_losses}")
    print(f"Strategy Win Rate:  {win_rate:.2f}%")
    print(f"Starting Capital:   ₹{STARTING_CAPITAL:,.2f}")
    print(f"Ending Capital:     ₹{current_capital:,.2f}")
    print(f"Net ROI (Post-Fee): {roi:+.2f}%")
    print("======================================")

# --- Execute Cell ---
run_smart_execution()

🤖 Booting Smart Execution Agent...
⚙️ Scanning 8,500 market environments for asymmetric Risk/Reward...

📈 ASYMMETRIC EXECUTION RESULTS 📈
Total Trades Taken: 128
Longs (W/L):        16 / 92
Shorts (W/L):       3 / 17
Strategy Win Rate:  14.84%
Starting Capital:   ₹10,000.00
Ending Capital:     ₹9,773.45
Net ROI (Post-Fee): -2.27%
